In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA

import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style='whitegrid')

In [2]:
df = pd.read_csv("daily_features.csv")

# Convert day column to datetime
df['date'] = pd.to_datetime(df['day'], errors='coerce')
df = df.sort_values('date').dropna(subset=['date'])

df.head()

,app_id,day,cost,value,data_used,data_sent,requests_made,cost_per_request_made,cost_per_request_received,data_per_request,data_sent_per_received,data_used_per_received,requests_per_business_hour,requests_per_hour,requests_received,value_per_cost,cost_per_request,value_per_cost.1,data_per_request.1,date
0,SELENE,2024-06-01,158758.25,4646.1,4646.10,4646.1,5490.0,28.917714,86.753142,0.846284,2.538852,2.538852,296.8,228.750000,1830.0,0.029265,28.917714,0.029265,0.846284,2024-06-01
63,MULTILNG,2024-06-01,3589.75,0.0,110.55,0.0,63.0,56.980159,0.000000,1.754762,0.000000,0.000000,2.7,2.625000,0.0,0.000000,56.980159,0.000000,1.754762,2024-06-01
62,JPNXPP,2024-06-01,3100.05,57.0,57.00,57.0,94.0,32.979255,65.958511,0.606383,1.212766,1.212766,5.1,3.916667,47.0,0.018387,32.979255,0.018387,0.606383,2024-06-01
61,ECOMLP,2024-06-01,28875.70,0.0,594.90,0.0,216.0,133.683796,0.000000,2.754167,0.000000,0.000000,8.0,9.000000,0.0,0.000000,133.683796,0.000000,2.754167,2024-06-01
60,GEOANL,2024-06-01,5129.40,99.6,157.20,99.6,87.0,58.958621,284.966667,1.806897,5.533333,8.733333,4.6,3.625000,18.0,0.019417,58.958621,0.019417,1.806897,2024-06-01


In [3]:
features = [
    'requests_made', 'cost', 'value', 'data_used', 
    'data_sent', 'cost_per_request_made', 'cost_per_request_received', 
    'data_per_request', 'data_sent_per_received', 'data_used_per_received'
]

X = df[features].fillna(0)  # fill missing with 0
X_scaled = StandardScaler().fit_transform(X)  # normalize features


In [4]:
sim_matrix = cosine_similarity(X_scaled)

# Convert to DataFrame for readability
sim_df = pd.DataFrame(sim_matrix, index=df['date'], columns=df['date'])
sim_df.head()


date,2024-06-01,2024-06-01,2024-06-01,2024-06-01,2024-06-01,2024-06-01,2024-06-01,2024-06-01,2024-06-01,2024-06-01,...,2025-05-31,2025-05-31,2025-05-31,2025-05-31,2025-05-31,2025-05-31,2025-05-31,2025-05-31,2025-05-31,2025-05-31
date,,,,,,,,,,,,,,,,,,,,,
2024-06-01,1.000000,-0.525304,-0.532584,0.170933,-0.056359,-0.550549,-0.668174,-0.559754,-0.420709,-0.662625,...,-0.337283,-0.103593,-0.350182,-0.522743,0.634606,-0.304227,0.425655,0.335398,-0.328524,0.002621
2024-06-01,-0.525304,1.000000,-0.038692,0.413048,-0.479416,0.512334,0.842625,0.449228,0.213269,0.869839,...,-0.160314,-0.567765,0.833481,-0.023979,-0.479664,-0.380340,-0.776312,-0.719083,0.809116,0.477005
2024-06-01,-0.532584,-0.038692,1.000000,-0.843087,0.002353,0.662729,0.239309,0.637323,0.848595,0.237115,...,0.931618,0.148298,-0.500375,0.998256,0.284149,0.886083,-0.161383,-0.238879,-0.526682,-0.818420
2024-06-01,0.170933,0.413048,-0.843087,1.000000,-0.171195,-0.325883,0.173854,-0.307868,-0.633716,0.178839,...,-0.833362,-0.333806,0.777836,-0.837329,-0.519512,-0.876922,-0.137461,-0.039037,0.805298,0.958428
2024-06-01,-0.056359,-0.479416,0.002353,-0.171195,1.000000,-0.670563,-0.652064,-0.667625,-0.506102,-0.649697,...,-0.165044,0.946815,-0.267847,-0.052117,-0.163480,0.158542,0.851198,0.886163,-0.282717,-0.071806


In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(sim_df, cmap='viridis')
plt.title("Cosine Similarity Between Days")
plt.show()

In [ ]:
k = 3  # choose number of clusters
kmeans = KMeans(n_clusters=k, random_state=42)
clusters = kmeans.fit_predict(X_scaled)

df['cluster'] = clusters
df[['date','cluster']].head(10)

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8,6))
sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=df['cluster'], palette='Set2', s=100)
plt.title("Clustering of Days (PCA-reduced)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

In [ ]:
cluster_summary = df.groupby('cluster')[features].mean()
cluster_summary